## Agrupamento - K-Means e Aplicações

---

Essa primeira apresentação do método e a construção do dataset é baseada no material de David Sheehan.

In [1]:

import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import silhouette_score
from sklearn import cluster, datasets, mixture
from sklearn.neighbors import kneighbors_graph

np.random.seed(844)
clust1 = np.random.normal(5, 2, (1000,2))
clust2 = np.random.normal(15, 3, (1000,2))
clust3 = np.random.multivariate_normal([17,3], [[1,0],[0,1]], 1000)
clust4 = np.random.multivariate_normal([2,16], [[1,0],[0,1]], 1000)
dataset1 = np.concatenate((clust1, clust2, clust3, clust4))

# we take the first array as the second array has the cluster labels
dataset2 = datasets.make_circles(n_samples=1000, factor=.5, noise=.05)[0]

# plot clustering output on the two datasets
def cluster_plots(set1, set2, colours1 = 'gray', colours2 = 'gray', 
                  title1 = 'Dataset 1',  title2 = 'Dataset 2'):
    fig,(ax1,ax2) = plt.subplots(1, 2)
    fig.set_size_inches(6, 3)
    ax1.set_title(title1,fontsize=14)
    ax1.set_xlim(min(set1[:,0]), max(set1[:,0]))
    ax1.set_ylim(min(set1[:,1]), max(set1[:,1]))
    ax1.scatter(set1[:, 0], set1[:, 1],s=8,lw=0,c= colours1)
    ax2.set_title(title2,fontsize=14)
    ax2.set_xlim(min(set2[:,0]), max(set2[:,0]))
    ax2.set_ylim(min(set2[:,1]), max(set2[:,1]))
    ax2.scatter(set2[:, 0], set2[:, 1],s=8,lw=0,c=colours2)
    fig.tight_layout()
    plt.show()

cluster_plots(dataset1, dataset2)

<Figure size 600x300 with 2 Axes>

Uma forma de associar os pontos aos grupos é com o algoritmo K-Means, o qual inicia pontos aleatórios e depois associa os pontos em um processo repetitivo, até que a movimentação entre os pontos e os centros seja pequena ou nenhuma.
No entanto, é possível perceber que para certas estruturas, o K-Means não tem desempenho satisfatório. Especificamente, o K-Means é muito útil para estruturas globulares.

In [ ]:
# implementing k-means clustering
kmeans_dataset1 = cluster.KMeans(n_clusters=4, max_iter=300, 
                                 init='k-means++',n_init=10).fit_predict(dataset1)
kmeans_dataset2 = cluster.KMeans(n_clusters=2, max_iter=300, 
                                 init='k-means++',n_init=10).fit_predict(dataset2)
print('Dataset1')
print(*["Cluster "+str(i)+": "+ str(sum(kmeans_dataset1==i)) for i in range(4)], sep='\n')
cluster_plots(dataset1, dataset2, kmeans_dataset1, kmeans_dataset2)

No entanto, mesmo em estruturas globulares, o K-Means é sensível à densidade dos grupos. Reduzindo a quantidade de amostras de alguns grupos anteriores, faz com que o algoritmo se perca em identificar exatamente as estruturas.

In [ ]:
kmeans_dataset1 = cluster.KMeans(n_clusters=4, max_iter=300, 
                                 init='k-means++',n_init=10).fit_predict(np.vstack([dataset1[:2080,:],
                                                                                    dataset1[3000:3080,:]]))
kmeans_dataset2 = cluster.KMeans(n_clusters=4, max_iter=300, 
                                 init='k-means++',n_init=10).fit_predict(np.vstack([dataset1[-2080:,],
                                                                                    dataset1[:80,]]))
cluster_plots(np.vstack([dataset1[:2080,],dataset1[3000:3080,]]), 
              np.vstack([dataset1[-2080:,],dataset1[:80,]]), 
              kmeans_dataset1, kmeans_dataset2,title1='', title2='')

In [ ]:
hc_dataset1 = cluster.AgglomerativeClustering(n_clusters=4, affinity='euclidean', 
                                              linkage='ward').fit_predict(dataset1)
hc_dataset2 = cluster.AgglomerativeClustering(n_clusters=2, affinity='euclidean', 
                                              linkage='average').fit_predict(dataset2)
print("Dataset 1")
print(*["Cluster "+str(i)+": "+ str(sum(hc_dataset1==i)) for i in range(4)], sep='\n')
cluster_plots(dataset1, dataset2, hc_dataset1, hc_dataset2)

In [ ]:
hc_dataset2 = cluster.AgglomerativeClustering(n_clusters=2, affinity='euclidean', 
                                              linkage='complete').fit_predict(dataset2)
connect = kneighbors_graph(dataset2, n_neighbors=5, include_self=False)
hc_dataset2_connectivity = cluster.AgglomerativeClustering(n_clusters=2, affinity='euclidean', 
                                              linkage='complete',connectivity=connect).fit_predict(dataset2)
cluster_plots(dataset2, dataset2,hc_dataset2,hc_dataset2_connectivity,
             title1='Without Connectivity', title2='With Connectivity')

Outras técnicas e outras dificuldades dos conjuntos de dados podem ser observados nesse artigo mais completo, de onde essa parte do material foi retirada: https://dashee87.github.io/data%20science/general/Clustering-with-Scikit-with-GIFs/

---

### Classificação utilizando o K-Means

A classificação utilizando o K-Means implica em representar uma amostra pela distância dela para todos os centros de grupos no espaço vetorial. Desse modo, a quantidade de atributos dá amostra passa a ser a distância da amostra para cada centro configurado no modelo do K-Means.

In [ ]:
from sklearn.datasets import load_digits, load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans

def clust(dataset_load, n_clusters):

    ds = dataset_load()
    X = ds.data
    y = ds.target

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=2019)

    model = LogisticRegression(random_state=42)
    model.fit(X_train, y_train)
    print('Sem clustering:', model.score(X_test, y_test))

    km = KMeans(n_clusters = n_clusters, random_state=42)
    X_train = km.fit_transform(X_train)
    X_test  = km.transform(X_test)

    model = LogisticRegression(random_state=42)
    model.fit(X_train, y_train)
    print('Com clustering:', model.score(X_test, y_test))
    

print('Digits dataset')
clust(load_digits, n_clusters = 3)
print()
print('Iris dataset')
clust(load_iris, n_clusters = 3)


---

### Quantização de cores com K-Means

E se fosse possível agrupar os tons de cores, e recolorir uma imagem apenas utilizando as centróides do K-Means, em vez de todos os tons de cores? Esse exemplo é baseado no material da documentação do Scikit-Learn: https://scikit-learn.org/stable/auto_examples/cluster/plot_color_quantization.html

Se o exemplo não funcionar nas máquinas do laboratório, tente executar em sua casa após instalar a biblioteca PIL.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances_argmin
from sklearn.datasets import load_sample_image
from sklearn.utils import shuffle

n_colors = 64 # varie o numero de cores

china = load_sample_image("china.jpg") # tente como flower.jpg tambem

# normaliza entre 0 e 1 pro plt não ter problema
china = np.array(china, dtype=np.float64) / 255

w, h, d = original_shape = tuple(china.shape)
image_array = np.reshape(china, (w * h, d))

image_array_sample = shuffle(image_array, random_state=0)[:1000]
kmeans = KMeans(n_clusters=n_colors, random_state=0).fit(image_array_sample)

labels = kmeans.predict(image_array)

def recreate_image(codebook, labels, w, h):
    d = codebook.shape[1]
    image = np.zeros((w, h, d))
    label_idx = 0
    for i in range(w):
        for j in range(h):
            image[i][j] = codebook[labels[label_idx]]
            label_idx += 1
    return image

plt.figure(1)
plt.clf()
plt.axis('off')
plt.title('Original image (96,615 colors)')
plt.imshow(china)

plt.figure(2)
plt.clf()
plt.axis('off')
plt.title('Quantized image (64 colors, K-Means)')
plt.imshow(recreate_image(kmeans.cluster_centers_, labels, w, h))